In [ ]:
// Nested tuples, the representation of abstract syntax trees; loaded once per session.
#load "../Support/Nested.fsx"

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

# Symbolic Differentiation 

In this notebook our goal is to implement *symbolic differentiation*.  Concretely, we want do implement a function `diff` that takes one argument:
  - The argument `expr` represents an *arithmetic expression*.
    Here an arithmetic expression is any string that is build from variable and numbers by application
    of any of the operator symbols "`+`", "`-`", "`*`", "`/`", and "`**`".
    The operator "`**`" represents exponentiation, i.e. an expression of the form 
    $a \texttt{**} b$ is interpreted as $a^b$.          
    Furthermore, if $e$ is an expression, then both $\exp(e)$ and $\ln(e)$ are expressions too.

The function call `diff(expr)` will then compute the derivative of `expr` with respect to the variable `x`.  For example, the function call 
`diff("x * exp(x)")` will compute the output
`1 * exp(x) + x * exp(x)` because we have:
$$ \frac{\mathrm{d}\;}{\mathrm{d}x} \bigl( x \cdot \mathrm{e}^x \bigr) = 1 \cdot \mathrm{e}^x + x \cdot \mathrm{e}^x. $$

This file shows the implementation of a program that can perform *symbolic differentiation* using `Larkita`.  The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{expr}    & \rightarrow & \;\texttt{expr}\; \texttt{'+'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{expr}\; \texttt{'-'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{product}                                  \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{product}\; \texttt{'*'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{product}\; \texttt{'/'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{factor}                                   \\[0.2cm]
  \texttt{factor}  & \rightarrow & \texttt{base} \;\texttt{'**'} \; \texttt{factor}    \\
                   & \mid        & \texttt{base}                                       \\[0.2cm]
  \texttt{base}    & \rightarrow & \texttt{'exp'} \; \texttt{'('} \; \texttt{expr} \;\texttt{')'}     \\
                   & \mid        & \texttt{'ln'} \; \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \texttt{'('} \; \texttt{expr} \;\texttt{')'}                     \\
                   & \mid        & \;\texttt{NUMBER}                                                \\
                   & \mid        & \;\texttt{'x'}                               
\end{array}
$$

## Specification of the Grammar and Parser

We use `Larkita` to define our EBNF grammar, which is stored in the file `diff.lark`.  The rules inherently
enforce mathematical precedence.  We include a
rule `unary` to handle unary minus correctly relative to exponentiation and multiplication.

The string literals of an alternative form the *head* of the tuple that is returned for this alternative, while
the other symbols become its children.  For example, the alternative `expr "+" product` yields the tuple
`("+", ast(expr), ast(product))`, in F# `Node [Leaf (Str "+"); ast(expr); ast(product)]`, and `base "**" unary`
yields `("**", ast(base), ast(unary))`.  An *alias* `-> name` replaces the head, so the alternative
`"-" unary -> neg` yields `("neg", ast(unary))` and the alternative `"exp" "(" expr ")" -> exp` yields
`("exp", ast(expr))`.  An alternative that consists of a single symbol, like
`product: unary`, returns the abstract syntax tree of this symbol.  The directive `%ignore "(", ")"` removes the
parentheses from the head of the alternative `"(" expr ")"`, which then returns the abstract syntax tree of
`expr`.

In [ ]:
#load "../Support/Larkita/Larkita.fsx"
open Larkita
open Nested

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "diff.lark")

We generate a parser for this grammar.  `Larkita` writes a description of the states of the LR(1) automaton into
the file `diff.out`.

In [ ]:
// The grammar is found next to this notebook, also when the exported module is loaded from elsewhere.
let parser = Larkita.Open(System.IO.Path.Combine(__SOURCE_DIRECTORY__, "diff.lark"), outFile = System.IO.Path.Combine(__SOURCE_DIRECTORY__, "diff.out"))

## AST Construction

The parser returns nested tuples whose heads are the operators:

In [ ]:
printfn "%s" (toString (parse parser "ln(x ** x) + exp(x * -x)"))

The function `convertLeaves` replaces the *leaves* of these tuples:  a number like `("NUMBER", "2")` becomes the
integer leaf `2`, i.e. `Leaf (Int 2)`, and the variable `x`, which is a string literal and therefore the tuple
`("x",)`, becomes the string `"x"`.  All other tuples are converted recursively, so `diff` works on nested tuples
like `("+", ("ln", "x"), 1)`.  The alias `-> neg` of the grammar yields the unary minus `("neg", f)`.  The helper
functions `op1` and `op2` build tuples consisting of an operator and one or two arguments.

In [ ]:
let op1 (op: string) (a: nested) = Node [ Leaf (Str op); a ]
let op2 (op: string) (a: nested) (b: nested) = Node [ Leaf (Str op); a; b ]

let rec convertLeaves (ast: nested) : nested =
    match ast with
    | Node [ Leaf (Str "NUMBER"); Leaf (Str digits) ] -> Leaf (Int (int digits))
    | Node [ Leaf (Str "x") ] -> Leaf (Str "x")
    | Node (op :: args) -> Node (op :: List.map convertLeaves args)
    | _ -> ast

let parseExpr source = convertLeaves (parse parser source)

In [ ]:
printfn "%A" (parseExpr "ln(x ** x) + exp(x * -x)")

Now we are ready to implement the function `diff` that takes an expression possibly containing the variable `x`.
It computes the derivative of the given expression with respect to `x`.
<ol>
<li>  The The corresponding cases implement the rule: 
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) + g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) + \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  The corresponding cases implement the rule:
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) - g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) - \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> is a product.  The 
      <a href="https://en.wikipedia.org/wiki/Product_rule">product rule</a> is      
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) \cdot g(x)\bigr) = \left(\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)\right)\cdot g(x) + f(x) \cdot \left(\frac{\mathrm{d}\;}{\mathrm{d}x} g(x)\right)
      $$
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> is a quotient.  The
      <a href="https://en.wikipedia.org/wiki/Quotient_rule">quotient rule</a> is
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x}\left(\frac{f(x)}{g(x)}\right) = 
         \frac{\displaystyle\left(\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)\right)\cdot g(x) - 
         f(x) \cdot \left(\frac{\mathrm{d}\;}{\mathrm{d}x} g(x)\right)}{g(x) \cdot g(x)}
      $$
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> is a power.  Now in order to take the derivative of an
      expression of the form
      $$  f(x)^{g(x)} $$
      we first need to rewrite this expression using the following trick:
      $$ f(x)^{g(x)} = \exp\bigl(\ln\bigl(f(x)^{g(x)}\bigr)\bigr) = \exp\bigl(g(x) \cdot \ln(f(x))\bigr) $$
      Then, we can recursively call <tt>diff</tt> for this expression.  This works, because the function
      <tt>diff</tt> can deal with both the exponential function $x \mapsto \exp(x)$ and with the natural
      logarithm $x \mapsto \ln(x)$.  This rewriting is done in The corresponding cases.
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> has the form 
      $$\ln\bigl(f(x)\bigr)$$  
      In order to take the derivative of this expression, we first need to know the derivative of the natural
      logarithm.  This derivative is given as     
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x} \ln(x) = \frac{1}{x}$$
      Then, using the <a href="https://en.wikipedia.org/wiki/Chain_rule">chain rule</a> we have that
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x} \ln\bigl(f(x)\bigr) = \frac{\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)}{f(x)}$$
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> has the form $\exp\bigl(f(x)\bigr)$.  
      In order to take the derivative of this expression, we first need to know the derivative of the 
      <a href="https://en.wikipedia.org/wiki/Exponential_function">exponential function</a>.  
      This derivative is given as 
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x} \exp(x) = \exp(x)$$    
      Then, using the <a href="https://en.wikipedia.org/wiki/Chain_rule">chain rule</a> we have that
      $$\frac{\mathrm{d}\;}{\mathrm{d}x} \exp\bigl(f(x)\bigr) = \left(\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)\right) \cdot \exp\bigl(f(x)\bigr)
      $$
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> is a variable and happens to be the same variable as
      <tt>x</tt>.   As we have
      $$\frac{\mathrm{d}x}{\mathrm{d}x} = 1,$$
      the function <tt>diff</tt> returns <tt>1</tt> in this case.
      </li>
<li>  Otherwise, the expression is assumed to be a constant and hence we return 0.
      </li>
</ol>

In [ ]:
let rec diff (e: nested) : nested =
    match e with
    | Node [ Leaf (Str "+"); f; g ] -> op2 "+" (diff f) (diff g)
    | Node [ Leaf (Str "-"); f; g ] -> op2 "-" (diff f) (diff g)
    | Node [ Leaf (Str "*"); f; g ] -> op2 "+" (op2 "*" (diff f) g) (op2 "*" f (diff g))
    | Node [ Leaf (Str "/"); f; g ] -> op2 "/" (op2 "-" (op2 "*" (diff f) g) (op2 "*" f (diff g))) (op2 "*" g g)
    | Node [ Leaf (Str "**"); f; g ] -> diff (op1 "exp" (op2 "*" g (op1 "ln" f)))
    | Node [ Leaf (Str "neg"); f ] -> op1 "neg" (diff f)
    | Node [ Leaf (Str "ln"); f ] -> op2 "/" (diff f) f
    | Node [ Leaf (Str "exp"); f ] -> op2 "*" (diff f) e
    | Leaf (Str "x") -> Leaf (Int 1)
    | _ -> Leaf (Int 0)

The function below turns the nested tuple into a readable expression and preserves precedence and associativity.

In [ ]:
let precedence (op: string) =
    match op with
    | "+" | "-" -> 1
    | "*" | "/" -> 2
    | "**" -> 4
    | _ -> 5

let precedenceOp (e: nested) =
    match e with
    | Node [ Leaf (Str "neg"); _ ] -> 3
    | Node [ Leaf (Str op); _; _ ] -> precedence op
    | _ -> 5

// Equal-precedence right children also need brackets for nonassociative operations.
let rec exprToString (expr: nested) =
    let wrap needed child =
        let text = exprToString child
        if needed then "(" + text + ")" else text
    match expr with
    | Leaf (Int n) -> string n
    | Leaf (Str s) -> s
    | Node [ Leaf (Str "neg"); argument ] -> "-" + wrap (precedenceOp argument < 3) argument
    | Node [ Leaf (Str f); argument ] -> f + "(" + exprToString argument + ")"
    | Node [ Leaf (Str op); left; right ] ->
        let p = precedence op
        let symbol = if op = "+" || op = "-" then " " + op + " " else op
        let leftNeeds = precedenceOp left < p || (op = "**" && precedenceOp left = p)
        let rightNeeds = precedenceOp right < p || (op <> "**" && precedenceOp right = p)
        wrap leftNeeds left + symbol + wrap rightNeeds right
    | _ -> failwithf "Unexpected tree %A" expr

let rec evaluate x (e: nested) =
    match e with
    | Leaf (Int n) -> float n
    | Leaf (Str "x") -> x
    | Node [ Leaf (Str "neg"); f ] -> -(evaluate x f)
    | Node [ Leaf (Str "exp"); f ] -> exp (evaluate x f)
    | Node [ Leaf (Str "ln"); f ] -> log (evaluate x f)
    | Node [ Leaf (Str op); f; g ] ->
        let a, b = evaluate x f, evaluate x g
        match op with
        | "+" -> a + b
        | "-" -> a - b
        | "*" -> a * b
        | "/" -> divide a b
        | _ -> a ** b
    | _ -> failwithf "Unexpected tree %A" e

In [ ]:
let test source =
    let expr = parseExpr source
    let derivative = diff expr
    printfn "d/dx %s = %s" source (exprToString derivative)
    // A numerical derivative independently checks the symbolic rules.
    let x, h = 1.7, 1e-6
    let numerical = (evaluate (x + h) expr - evaluate (x - h) expr) / (2.0 * h)
    checkNotebook (abs(evaluate x derivative - numerical) < 1e-4)
    derivative

In [ ]:
test "x ** x" |> ignore

In [ ]:
test "x ** -x" |> ignore

In [ ]:
test "-x ** 2" |> ignore

In [ ]:
test "1/(1+exp(-x))" |> ignore

In [ ]:
test "ln(exp(x)) / exp(x/x)" |> ignore
let grouped = parseExpr "x / (x * x)"
checkNotebook (parseExpr (exprToString grouped) = grouped)
let negated = parseExpr "-(x * x) ** 2"
checkNotebook (parseExpr (exprToString negated) = negated)

## Inspect the derivative tree

The derivative is a nested tuple, too.  It can be drawn as a tree to inspect product, chain and power rules.

In [ ]:
#load "../Support/Visualization.fsx"
open Visualization
let derivative = parseExpr "x ** x" |> diff
let svg = derivative |> treeToDot |> renderDot
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(svg), "text/html") |> ignore